In [1]:
from pathlib import Path
import subprocess, os

REPO_HINT = Path("/Users/mkrishan/Documents/boolearn-main")  # set to None for public release

def find_repo_root():
    candidates = []
    if REPO_HINT is not None:
        candidates.append(Path(REPO_HINT).expanduser().resolve())
    cwd = Path.cwd().resolve()
    candidates += [cwd] + list(cwd.parents)
    home = Path.home()
    candidates += [home/"boolearn-main", home/"Documents"/"boolearn-main", home/"Desktop"/"boolearn-main"]
    for p in candidates:
        if (p/"boolearn"/"src"/"makefile").exists():
            return p
    raise RuntimeError(f"❌ Could not locate repo root from {cwd}. Set REPO_HINT.")

ROOT = find_repo_root()
SRC  = ROOT / "boolearn" / "src"
DATA = ROOT / "data"
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

print("✅ ROOT:", ROOT)
print("✅ SRC:", SRC)
print("✅ DATA:", DATA)
print("✅ RESULTS:", RESULTS)

subprocess.run(["make","all"], cwd=str(SRC), check=True)
os.environ["PATH"] = f"{SRC}:{os.environ.get('PATH','')}"
print("✅ PATH updated with boolearn/src")

✅ ROOT: /Users/mkrishan/Documents/boolearn-main
✅ SRC: /Users/mkrishan/Documents/boolearn-main/boolearn/src
✅ DATA: /Users/mkrishan/Documents/boolearn-main/data
✅ RESULTS: /Users/mkrishan/Documents/boolearn-main/results
make: Nothing to be done for `all'.
✅ PATH updated with boolearn/src


In [3]:
from pathlib import Path
import subprocess

train   = SRC / "train"
layered = SRC / "layered"
assert train.exists(), train
assert layered.exists(), layered

dat = DATA / "decode" / "rand16.dat"
assert dat.exists(), dat

OUT = RESULTS / "03_decoder_rand16"
NET_DIR = OUT / "nets"
RUN_DIR = OUT / "runs"
NET_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("✅ train:", train)
print("✅ layered:", layered)
print("✅ dat:", dat)
print("✅ NET_DIR:", NET_DIR)
print("✅ RUN_DIR:", RUN_DIR)

✅ train: /Users/mkrishan/Documents/boolearn-main/boolearn/src/train
✅ layered: /Users/mkrishan/Documents/boolearn-main/boolearn/src/layered
✅ dat: /Users/mkrishan/Documents/boolearn-main/data/decode/rand16.dat
✅ NET_DIR: /Users/mkrishan/Documents/boolearn-main/results/03_decoder_rand16/nets
✅ RUN_DIR: /Users/mkrishan/Documents/boolearn-main/results/03_decoder_rand16/runs


In [5]:
import subprocess

wth = NET_DIR / "rand16.wth"
net = NET_DIR / "rand16.net"

# Decoder net from the guide:
# 2
# 4 16 16
with open(wth, "w") as f:
    f.write("2\n4 16 16\n")

print("Wrote:", wth)
print(wth.read_text())

if not net.exists():
    cmd = [str(layered), "0.5", str(wth), str(net)]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=str(NET_DIR), check=False)

assert net.exists() and net.stat().st_size > 0
print("✅ net:", net, "| bytes:", net.stat().st_size)
with open(net) as f:
    print("Header:", f.readline().strip())

Wrote: /Users/mkrishan/Documents/boolearn-main/results/03_decoder_rand16/nets/rand16.wth
2
4 16 16

Running: /Users/mkrishan/Documents/boolearn-main/boolearn/src/layered 0.5 /Users/mkrishan/Documents/boolearn-main/results/03_decoder_rand16/nets/rand16.wth /Users/mkrishan/Documents/boolearn-main/results/03_decoder_rand16/nets/rand16.net
✅ net: /Users/mkrishan/Documents/boolearn-main/results/03_decoder_rand16/nets/rand16.net | bytes: 8815
Header: 37  5  16  352


In [11]:
import subprocess, time, select
from pathlib import Path

run_id = "rand16_demo"
gap_path = RUN_DIR / f"{run_id}.gap"

items  = "16"
btfn   = "3.0"
beta   = "0.5"
gam    = "1e-4"
epochs = "80"
itermax= "500000"
gapstop= "0.01"
trials = "1"

cmd = [str(train), str(net), str(dat),
       items, btfn, beta, gam, epochs, itermax, gapstop, trials, run_id]

print("Running:", " ".join(cmd))
print("Watching:", gap_path.name)

proc = subprocess.Popen(
    cmd,
    cwd=str(RUN_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

last_lines = 0
printed_header = False

try:
    while True:
        # train is often quiet; avoid blocking
        if proc.stdout is not None:
            r, _, _ = select.select([proc.stdout], [], [], 0.0)
            if r:
                _ = proc.stdout.readline()

        if gap_path.exists():
            lines = [ln.strip() for ln in gap_path.read_text().splitlines() if ln.strip()]

            if lines and not printed_header:
                print("\niter\tgap_total\tbest_gap")
                printed_header = True

            if len(lines) > last_lines:
                for ln in lines[last_lines:]:
                    parts = ln.split()
                    # gap_total=col5, best_gap=col6 in guide
                    it = parts[0]
                    gap_total = parts[4]
                    best_gap  = parts[5]
                    print(f"{it}\t{gap_total}\t{best_gap}")
                last_lines = len(lines)

        if proc.poll() is not None:
            break

        time.sleep(0.2)

finally:
    try: proc.kill()
    except Exception: pass

print("\nReturn code:", proc.returncode)
print("\nArtifacts:")
for p in sorted(RUN_DIR.glob(run_id + ".*")):
    print(" -", p.name, "|", p.stat().st_size, "bytes")

Running: /Users/mkrishan/Documents/boolearn-main/boolearn/src/train /Users/mkrishan/Documents/boolearn-main/results/03_decoder_rand16/nets/rand16.net /Users/mkrishan/Documents/boolearn-main/data/decode/rand16.dat 16 3.0 0.5 1e-4 80 500000 0.01 1 rand16_demo
Watching: rand16_demo.gap

iter	gap_total	best_gap
2	2.09731037	2.09731037
3	1.38831924	1.38831924
4	1.07261231	1.07261231
5	1.04900339	1.04900339
6	1.10962633	1.04900339
7	1.02093557	1.02093557
8	1.03000591	1.02093557
9	1.00692854	1.00692854
10	0.96440839	0.96440839
11	1.00312750	0.96440839
12	0.91842266	0.91842266
13	0.88613552	0.88613552
14	0.86945982	0.86945982
15	0.87730834	0.86945982
16	0.95040053	0.86945982
17	0.89645136	0.86945982
18	0.85088365	0.85088365
20	0.83189116	0.83189116
23	0.82227830	0.80227105
27	0.73056583	0.70443087
32	0.73367695	0.66846541
37	0.65940772	0.65500189
44	0.65021110	0.61709924
52	0.61000316	0.60890880
61	0.62660052	0.60890880
72	0.59523688	0.54531220
84	0.54410445	0.50637198
99	0.53292664	0.49897261

In [13]:
from pathlib import Path

gen_path = RUN_DIR / f"{run_id}.gen"
print("gen exists?", gen_path.exists(), gen_path)

if gen_path.exists():
    print("\n=== head(gen) ===")
    with open(gen_path) as f:
        for _ in range(30):
            line = f.readline()
            if not line:
                break
            print(line.rstrip())
else:
    print("⚠️ No .gen file produced. If gap stayed large, increase itermax or tweak beta/gam.")

gen exists? True /Users/mkrishan/Documents/boolearn-main/results/03_decoder_rand16/runs/rand16_demo.gen

=== head(gen) ===
16  4

 0 0 0 0	1
 1 0 0 0	1
 0 1 0 0	1
 1 1 0 0	1
 0 0 1 0	1
 1 0 1 0	1
 0 1 1 0	1
 1 1 1 0	1
 0 0 0 1	1
 1 0 0 1	1
 0 1 0 1	1
 1 1 0 1	1
 0 0 1 1	1
 1 0 1 1	1
 0 1 1 1	1
 1 1 1 1	1
